In [0]:
%sql 
create volume if not exists spark_lab

In [0]:
import requests
catalog_name=spark.sql("SELECT current_catalog()").collect()[0][0]
print(catalog_name)

dbws01


In [0]:
volume_base=f"/Volumes/{catalog_name}/default/spark_lab"

files=["2019_edited.csv", "2020_edited.csv","2021_edited.csv"]

for file in files:
  url=f"https://raw.githubusercontent.com/kuljotSB/DatabricksGenAIEngineer/refs/heads/main/Databricks_Fundamentals/{file}/Volumes/dbws01/default/spark_lab/2019_edited.csv/Volumes/dbws01/default/spark_lab/2019_edited.csv"
  r = requests.get(url)
  with open(f"{volume_base}/{file}", "wb") as code:
    code.write(r.content)


In [0]:
#load csv into Dataframes
df=spark.read.load(f'/Volumes/{catalog_name}/default/spark_lab/*_edited.csv', format='csv')
df.show()

+-------+---+----------+-------------------+--------------------+--------------------+---+---------+--------+
|    _c0|_c1|       _c2|                _c3|                 _c4|                 _c5|_c6|      _c7|     _c8|
+-------+---+----------+-------------------+--------------------+--------------------+---+---------+--------+
|SO49171|  1|2021-01-01|      Mariah Foster|mariah21@adventur...|  Road-250 Black, 48|  1|2181.5625| 174.525|
|SO49172|  1|2021-01-01|       Brian Howard|brian23@adventure...|    Road-250 Red, 44|  1|  2443.35| 195.468|
|SO49173|  1|2021-01-01|      Linda Alvarez|linda19@adventure...|Mountain-200 Silv...|  1|2071.4196|165.7136|
|SO49174|  1|2021-01-01|     Gina Hernandez|gina4@adventure-w...|Mountain-200 Silv...|  1|2071.4196|165.7136|
|SO49178|  1|2021-01-01|          Beth Ruiz|beth4@adventure-w...|Road-550-W Yellow...|  1|1000.4375|  80.035|
|SO49179|  1|2021-01-01|          Evan Ward|evan13@adventure-...|Road-550-W Yellow...|  1|1000.4375|  80.035|
|SO49175| 

In [0]:
from pyspark.sql.types import *
schema=StructType([
    StructField('SalesOrderNumber',StringType()),
    StructField("SalesorderLineNumber",IntegerType()),
    StructField("OrderDate",DateType()),
    StructField("CustomerName", StringType()),
    StructField("Email",StringType()),
    StructField("Item",StringType()),
    StructField("Quantity", IntegerType()),
    StructField("UnitPrice", FloatType()),
    StructField("Tax", FloatType())

])


In [0]:
bronze_df=spark.read.load(f'/Volumes/{catalog_name}/default/spark_lab/*_edited.csv', format='csv',schema=schema)


In [0]:
%sql
create schema if not exists dbws01.bronze

In [0]:
bronzeDeltaTablePath=f"/Volumes/dbws01/default/spark_lab/delta/bronze_sales_orders"

bronze_df.write.format('delta').mode("overwrite").save(bronzeDeltaTablePath)

bronze_df.write.format('delta').saveAsTable('bronze.bronze_sales_orders')

In [0]:
#silver table
silver_df=spark.read.format('delta').table('bronze.bronze_sales_orders')
silver_df.show(10)

+----------------+--------------------+----------+----------------+--------------------+--------------------+--------+---------+--------+
|SalesOrderNumber|SalesorderLineNumber| OrderDate|    CustomerName|               Email|                Item|Quantity|UnitPrice|     Tax|
+----------------+--------------------+----------+----------------+--------------------+--------------------+--------+---------+--------+
|         SO43701|                   1|2019-07-01|     Christy Zhu|christy12@adventu...|Mountain-100 Silv...|       1|  3399.99|271.9992|
|         SO43704|                   1|2019-07-01|      Julio Ruiz|julio1@adventure-...|Mountain-100 Blac...|       1|  3374.99|269.9992|
|         SO43705|                   1|2019-07-01|       Curtis Lu|curtis9@adventure...|Mountain-100 Silv...|       1|  3399.99|271.9992|
|         SO43700|                   1|2019-07-01|    Ruben Prasad|ruben10@adventure...|  Road-650 Black, 62|       1| 699.0982| 55.9279|
|         SO43703|                

In [0]:
from pyspark.sql.functions import *
silver_df=silver_df.dropDuplicates()
silver_df=silver_df.withColumn('Tax',col('UnitPrice')*0.08)
silver_df=silver_df.withColumn('Tax', col('Tax').cast("float"))

In [0]:
%sql 
create schema if not exists dbws01.silver

In [0]:
silverDeltaTablepath=f'/Volumes/dbws01/default/spark_lab/delta/silver_sales_orders'
silver_df.write.format('delta').mode('overwrite').save(silverDeltaTablepath)

#storing silver layer as a delta table in the catalog
silver_df.write.format('delta').saveAsTable('silver.silver_sales_orders')

In [0]:
#gold layer 
gold_df=spark.read.format("delta").table("silver.silver_sales_orders")
yearlySales=gold_df.select(year('OrderDate').alias('year')).groupBy("Year").count().orderBy("Year")
display(yearlySales)

Year,count
2019,1201
2020,2733
2021,28784


In [0]:
%sql --create gold schema 
create schema if not exists dbws01.gold

In [0]:
goldTablePath=f'/Volumes/dbws01/default/spark_lab/delta/gold_sales_orders'
yearlySales.write.format('delta').mode('overwrite').save(goldTablePath)

#storing gold layer as a delta table in the catalog
yearlySales.write.format('delta').saveAsTable('gold.gold_sales_orders')